In [ ]:
import cv2
import numpy as np
import pandas as pd
from skimage.feature import local_binary_pattern
from skimage.filters import gabor
from tqdm import tqdm

# --- Load the region proposals CSV (already generated earlier) ---
regions_df = pd.read_csv("region_proposals_step22.csv")

# --- Initialize SIFT and BoVW vocabulary ---
sift = cv2.SIFT_create()
bow_trainer = cv2.BOWKMeansTrainer(100)
bow_extractor = cv2.BOWImgDescriptorExtractor(sift, cv2.BFMatcher(cv2.NORM_L2))

# --- Step 1: Collect descriptors for BoVW Vocabulary Training ---
print("[INFO] Collecting SIFT descriptors for BoVW vocabulary...")
for img_path in tqdm(regions_df["image_path"].unique()):
    image = cv2.imread(img_path)
    if image is None:
        continue
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    if descriptors is not None:
        bow_trainer.add(descriptors)

# --- Step 2: Cluster descriptors into visual words (BoVW codebook) ---
vocab = bow_trainer.cluster()
bow_extractor.setVocabulary(vocab)
np.save("bovw_vocab.npy", vocab)
print(f"[INFO] BoVW vocabulary created with {len(vocab)} clusters.")

# --- Helper functions ---
def extract_sift_bovw(img_gray):
    keypoints = sift.detect(img_gray, None)
    if not keypoints:
        return np.zeros((len(vocab),))
    features = bow_extractor.compute(img_gray, keypoints)
    return features.flatten() if features is not None else np.zeros((len(vocab),))

def extract_gabor_features(img_gray):
    responses = []
    for theta in np.arange(0, np.pi, np.pi / 4):
        for frequency in (0.1, 0.3, 0.5):
            filt_real, filt_imag = gabor(img_gray, frequency=frequency, theta=theta)
            responses.extend([filt_real.mean(), filt_real.var()])
    return np.array(responses)

def extract_lbp_features(img_gray):
    lbp = local_binary_pattern(img_gray, P=8, R=1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 59), range=(0, 58))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    return hist

def extract_hsv_histogram(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist

# --- Step 3: Extract and fuse all features per region ---
fused_features = []

print("[INFO] Extracting features and fusing them per region...")
for idx, row in tqdm(regions_df.iterrows(), total=len(regions_df)):
    img = cv2.imread(row["image_path"])
    if img is None:
        continue
    x1, y1, x2, y2 = int(row["x1"]), int(row["y1"]), int(row["x2"]), int(row["y2"])
    region = img[y1:y2, x1:x2]
    if region.size == 0:
        continue

    gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)

    sift_vec = extract_sift_bovw(gray)
    gabor_vec = extract_gabor_features(gray)
    lbp_vec = extract_lbp_features(gray)
    hsv_vec = extract_hsv_histogram(region)

    fused = np.hstack([sift_vec, gabor_vec, lbp_vec, hsv_vec])
    fused_features.append(fused)

# --- Step 4: Save fused features ---
fused_features = np.array(fused_features)
np.save("fused_features.npy", fused_features)

fused_df = pd.DataFrame(fused_features)
fused_df.to_csv("fused_features.csv", index=False)

print(f"[INFO] Feature extraction complete. Saved {len(fused_features)} fused feature vectors.")
